# Document retrieval and Reranking

En este cuaderno cargaremos documentos de un pdf, haremos el chunking, lo guardaremos en una vectorstore y haremos el último paso, el prompt augmentation.


## Install dependencies

In [2]:
!pip install -U langchain langchain-community langchain-cohere pypdf openai langchain_experimental langchain_openai faiss-cpu tiktoken

## Document Loading



Cargaremos la transcripción de un curso de Andrew Ng: CS229 course

[Stanford CS229: Machine Learning Course, Lecture 1 - Andrew Ng (Autumn 2018)](https://see.stanford.edu/materials/aimlcs229/transcripts/MachineLearning-Lecture01.pdf)

In [3]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("https://see.stanford.edu/materials/aimlcs229/transcripts/MachineLearning-Lecture01.pdf")
lecture1_pages = loader.load()

In [4]:
len(lecture1_pages)

22

In [5]:
page = lecture1_pages[0]

In [6]:
print(page.page_content[0:500])

MachineLearning-Lecture01  
Instructor (Andrew Ng): Okay. Good morning. Welcome to CS229, the machine 
learning class. So what I wanna do today is just spend a little time going over the logistics 
of the class, and then we'll start to talk a bit about machine learning.  
By way of introduction, my name's Andrew Ng and I'll be instructor for this class. And so 
I personally work in machine learning, and I've worked on it for about 15 years now, and 
I actually think that machine learning is the 


In [7]:
page.metadata

{'producer': 'Acrobat Distiller 8.1.0 (Windows)',
 'creator': 'PScript5.dll Version 5.2.2',
 'creationdate': '2008-07-11T11:25:23-07:00',
 'author': '',
 'moddate': '2008-07-11T11:25:23-07:00',
 'title': '',
 'source': 'https://see.stanford.edu/materials/aimlcs229/transcripts/MachineLearning-Lecture01.pdf',
 'total_pages': 22,
 'page': 0,
 'page_label': '1'}

## Document Chunking

In [8]:
!pip install -U langchain-text-splitters

In [9]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)

In [10]:
docs = text_splitter.split_documents(lecture1_pages)

In [11]:
len(docs)

78

In [12]:
print(docs[0])

page_content='MachineLearning-Lecture01  
Instructor (Andrew Ng): Okay. Good morning. Welcome to CS229, the machine 
learning class. So what I wanna do today is just spend a little time going over the logistics 
of the class, and then we'll start to talk a bit about machine learning.  
By way of introduction, my name's Andrew Ng and I'll be instructor for this class. And so 
I personally work in machine learning, and I've worked on it for about 15 years now, and 
I actually think that machine learning is the most exciting field of all the computer 
sciences. So I'm actually always excited about teaching this class. Sometimes I actually 
think that machine learning is not only the most exciting thing in computer science, but 
the most exciting thing in all of human endeavor, so maybe a little bias there.  
I also want to introduce the TAs, who are all graduate students doing research in or 
related to the machine learning and all aspects of machine learning. Paul Baumstarck' metadata={'

## Retrieval

In [13]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

Enter your OpenAI API Key:··········


In [14]:
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings

vectorstore = FAISS.from_documents(docs, embedding=OpenAIEmbeddings(model="text-embedding-3-small"))

In [15]:
query = "Can you tell me something about the honor code?"

In [16]:
docs_and_scores = vectorstore.similarity_search_with_score(query, k = 10)

In [17]:
for doc in docs_and_scores:
  print(doc)

(Document(id='d76db6bb-f7b5-4ba9-90d2-f363deada8c6', metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2008-07-11T11:25:23-07:00', 'author': '', 'moddate': '2008-07-11T11:25:23-07:00', 'title': '', 'source': 'https://see.stanford.edu/materials/aimlcs229/transcripts/MachineLearning-Lecture01.pdf', 'total_pages': 22, 'page': 6, 'page_label': '7'}, page_content="that people that have taken this class in previous years may have written out by \nthemselves, okay?  \nSadly, in this class, there are usually — sadly, in previous years, there have often been a \nfew honor code violations in this class. And last year, I think I prosecuted five honor code \nviolations, which I think is a ridiculously large number. And so just don't work without \nsolutions, and hopefully there'll be zero honor code violations this year. I'd love for that \nto happen.  \nThe section here on the late homework policy if you ever want to hand in a hom

> ⚠️ **Atención · abril 2026**: la sección que sigue está etiquetada como "Rerank" pero **NO es un reranker real** · solo ordena los resultados del retriever por su score. Se mantiene por valor pedagógico (ver el contraste). El reranker de verdad (cross-encoder con Cohere `rerank-v3.5`) está al final del notebook.

## Rerank

In [18]:
filtered_documents = filter(lambda x: x[1] < 0.4, docs_and_scores)

In [19]:
sorted_documents = sorted(filtered_documents, key=lambda x: x[1], reverse=True)

In [20]:
for doc in sorted_documents:
  print(doc)

## Prompt Augmentation

In [21]:
from langchain_core.prompts import ChatPromptTemplate

rag_template = """\
Use the following context to answer the user's query. If you cannot answer, please respond with 'I don't know'.
Don't invent anything. Use only the Context, do not use your own knowledge.

User's Query:
***
{question}
***

Context:
***
{context}
***
"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

In [22]:
chunk_retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = { "k": 5 }  # antes: score_threshold=0.4 (brittle, dependía de la escala de ada-002)
)

In [23]:
chunk_retriever.invoke("Can you tell me something about the honor code?")

[Document(id='d76db6bb-f7b5-4ba9-90d2-f363deada8c6', metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2008-07-11T11:25:23-07:00', 'author': '', 'moddate': '2008-07-11T11:25:23-07:00', 'title': '', 'source': 'https://see.stanford.edu/materials/aimlcs229/transcripts/MachineLearning-Lecture01.pdf', 'total_pages': 22, 'page': 6, 'page_label': '7'}, page_content="that people that have taken this class in previous years may have written out by \nthemselves, okay?  \nSadly, in this class, there are usually — sadly, in previous years, there have often been a \nfew honor code violations in this class. And last year, I think I prosecuted five honor code \nviolations, which I think is a ridiculously large number. And so just don't work without \nsolutions, and hopefully there'll be zero honor code violations this year. I'd love for that \nto happen.  \nThe section here on the late homework policy if you ever want to hand in a hom

In [24]:
from langchain_openai import ChatOpenAI

base_model = ChatOpenAI(model = "gpt-5-nano")

In [25]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context" : chunk_retriever, "question" : RunnablePassthrough()}
    | rag_prompt
    | base_model
    | StrOutputParser()
)

In [26]:
rag_chain.invoke("Can you tell me something about the honor code?")

'- The instructor mentions the honor code explicitly.\n- There have been honor code violations in this class in previous years; last year five violations were prosecuted.\n- He encourages forming study groups and working together on homework, but stresses to avoid violating the honor code (e.g., “don’t work without solutions”).'

In [27]:
rag_chain.invoke("Can you tell me something about the sex code?")

"I don't know."

In [28]:
rag_chain.invoke("Who is Leo Messi?")

"I don't know"

In [29]:
rag_chain.invoke("Can you explain Transformers in LLMs?")

"I don't know"

In [30]:
rag_chain.invoke("Can you explain Learning Algorithms?")

'Learning algorithms are methods that allow computers to learn from data, so they can do tasks that are hard to program by hand. They’re used to infer patterns, make predictions, or automate decisions.\n\nKey points from the context:\n- They can learn what you like (e.g., recommendations for movies or purchases) to customize what you’re offered.\n- They can optimize things like driving performance for fuel efficiency.\n- They’re used in fields like handwriting/digit recognition (handwritten digits) and data mining on large medical records to derive medical knowledge.\n- They’re involved in everyday tasks you may not notice, such as automatically reading ZIP codes on mail.\n\nAnalogies and goals:\n- An analogy: tools (like a hammer and nails) plus a skilled practitioner. Learning algorithms are powerful tools, but mastery comes from understanding how to apply them well, not just having the tools.\n- The course also covers questions from learning theory, such as how much training data is

## Reranker real · Cohere `rerank-v3.5` (añadido abril 2026)

La sección anterior etiquetaba como "reranker" un simple sort por score · eso **no es un reranker**, es un retriever ordenado.

Un reranker de verdad es un **cross-encoder**: mira `(query, doc)` juntos y predice una score de relevancia mucho más precisa que la similitud coseno del bi-encoder.

Patrón productivo: retrieval amplio (top-20) + rerank fino (top-3).

In [31]:
!pip install -U langchain langchain-community langchain-cohere

In [34]:
import os, getpass

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key: ")

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

compressor = CohereRerank(
    model="rerank-v3.5",
    top_n=3,
)

rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

Mejora típica con un reranker bueno: **+10 a +20 puntos** en métricas como MRR o NDCG. Es el cambio más coste-efectivo que puedes hacer en un RAG (1 línea de código).